In [398]:
import os
import sys
import pandas as pd

# Resolve repo root whether cwd is repo root, scripts/, or a stage subdirectory.
cwd = os.getcwd()
if os.path.basename(cwd) == "orb-selection":
    repo_root = cwd
elif os.path.basename(os.path.dirname(cwd)) == "orb-selection":
    repo_root = os.path.dirname(cwd)
elif os.path.basename(os.path.dirname(os.path.dirname(cwd))) == "orb-selection":
    repo_root = os.path.dirname(os.path.dirname(cwd))
else:
    repo_root = cwd

src_path = os.path.join(repo_root, "src")
stage03_path = os.path.join(repo_root, "scripts", "03_selection_tests")
stage04_path = os.path.join(repo_root, "scripts", "04_permulation_loss_dup")
for path in (src_path, stage03_path, stage04_path):
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Using src path: {src_path}")
print(f"Using stage-03 path: {stage03_path}")
print(f"Using stage-04 path: {stage04_path}")

# Import modules
import odds_ratio_test as ort
from hyphy_results_parser import (
    RelaxResult,
    BustedPhResult
)

hyphy_results = os.path.join(repo_root, "results", "hyphy_results_cache")

%load_ext autoreload

Using src path: /Users/calvin/orb-selection/src
Using stage-03 path: /Users/calvin/orb-selection/scripts/03_selection_tests
Using stage-04 path: /Users/calvin/orb-selection/scripts/04_permulation_loss_dup
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load BLAST best hits for all N5 HOGs (occ>=30) in Ptep and Udiv genomes

In [399]:
N5_udiv_blasted_df = pd.read_csv(f"{repo_root}/data/N5_occ_30_blasted_udiv.tsv", sep="\t", index_col=0)
N5_ptep_blasted_df = pd.read_csv(f"{repo_root}/data/N5_occ_30_blasted_ptep.tsv", sep="\t", index_col=0)

In [400]:
def remove_dup_and_no_hits(loc_list_name):
    """Remove duplicates and 'no hits' from a list of LOCs."""

    loc_list = globals()[loc_list_name]
    #filter out "NO_HIT" entries from the lists
    loc_list = [item for item in loc_list if item != "NO_HIT"]

    #remove any items that do not start with "LOC" from the lists
    loc_list = [item for item in loc_list if item.startswith("LOC")]
    #delete the "LOC" prefix from the locs in the lists
    loc_list = [item[3:] for item in loc_list]
    #remove duplicates from the lists
    globals()[loc_list_name] = list(set(loc_list))
    print(f"{loc_list_name}: {len(globals()[loc_list_name] )}")


# HyPhy

In [401]:
# Load the saved RELAX results
relax_result = RelaxResult.load_from_pickle(os.path.join(hyphy_results, "relax_results.pkl"))

# Load the saved BUSTED-PH results
busted_ph_orb_result = BustedPhResult.load_from_pickle(os.path.join(hyphy_results, "busted_ph_orb_results.pkl"))

# Load the saved BUSTED-PH-REV results
busted_ph_non_orb_result = BustedPhResult.load_from_pickle(os.path.join(hyphy_results, "busted_ph_non_orb_results.pkl"))

In [402]:
for result in ("busted_ph_orb", "busted_ph_non_orb"):
    globals()[f"{result}_universe"] = globals()[f"{result}_result"].filter_omega(10000)
    globals()[f"{result}_universe_hogs"] = globals()[f"{result}_universe"].results_df.index.tolist()
    print(f"{result}_universe_hogs: {len(globals()[f'{result}_universe_hogs'])} unique items after filtering by omega < 10000")

    globals()[f"{result}_hogs"] = globals()[f"{result}_universe"].get_significant_results(alpha=0.05).index.tolist()

    print(f"{result}_hogs: {len(globals()[f'{result}_hogs'])} significant items at alpha = 0.05\n")

busted_ph_orb_universe_hogs: 4604 unique items after filtering by omega < 10000
busted_ph_orb_hogs: 93 significant items at alpha = 0.05

busted_ph_non_orb_universe_hogs: 4556 unique items after filtering by omega < 10000
busted_ph_non_orb_hogs: 235 significant items at alpha = 0.05



In [403]:
relax_universe = relax_result.filter_omega(10000)
relaxed_universe_hogs = relax_universe.results_df.index.tolist()
intensified_universe_hogs = relaxed_universe_hogs
print(f"relaxed_universe_hogs: {len(relaxed_universe_hogs)} unique items after filtering by omega < 10000")

relax_all_hogs = relax_universe.get_significant_results(alpha=0.05)
relaxed_hogs = relax_all_hogs[relax_all_hogs["result"]=="relaxed"].index.tolist()
intensified_hogs = relax_all_hogs[relax_all_hogs["result"]=="intensified"].index.tolist()

print(f"all_hogs: {len(relax_all_hogs)} significant items at alpha = 0.05")
print(f"relaxed_hogs: {len(relaxed_hogs)} significant items at alpha = 0.05")
print(f"intensified_hogs: {len(intensified_hogs)} significant items at alpha = 0.05")

relaxed_universe_hogs: 4625 unique items after filtering by omega < 10000
all_hogs: 1142 significant items at alpha = 0.05
relaxed_hogs: 405 significant items at alpha = 0.05
intensified_hogs: 737 significant items at alpha = 0.05


# Odds Ratio Test

In [404]:
%autoreload 2
results_all = ort.PermulationTestResults.load_from_pickle("results/odds_ratio_test/Results_Aug12/Run1_occ_30-88_10000x/results.pkl")

In [405]:
dup_res = results_all.results_df[results_all.results_df["Occupancy"]>=30][["Occupancy", "Log odds ratio of duplication", "P-value duplication more likely in fg", "P-value duplication more likely in bg"]]
dup_res.rename(columns={"Log odds ratio of duplication": "LOR",
                   "P-value duplication more likely in fg": "pval_fg",
                   "P-value duplication more likely in bg": "pval_bg"}, inplace=True)

In [406]:
loss_res = results_all.results_df[(results_all.results_df["Occupancy"]>=30) & (results_all.results_df["Occupancy"]<=88)][["Occupancy", "Log odds ratio of loss", "P-value loss more likely in fg", "P-value loss more likely in bg"]]
loss_res.rename(columns={"Log odds ratio of loss": "LOR",
                   "P-value loss more likely in fg": "pval_fg",
                   "P-value loss more likely in bg": "pval_bg"}, inplace=True)

In [407]:
dup_fg_universe_hogs = list(dup_res.index)
dup_bg_universe_hogs = list(dup_res.index)

loss_fg_universe_hogs = list(loss_res.index)
loss_bg_universe_hogs = list(loss_res.index)

In [408]:
for df in ["loss_fg", "loss_bg", "dup_fg", "dup_bg"]:
    hogs = list(results_all.results_fltrd_dfs[df].index)

    # filter by avg ci
    mask = results_all.results_fltrd_dfs[df]["Significant by avgd thresholds"].str.contains(df, na=False)
    hogs_fltrd = results_all.results_fltrd_dfs[df].index[mask].tolist()

    globals()[f"{df}_hogs"] = hogs
    globals()[f"{df}_fltrd_hogs"] = hogs_fltrd
    print(f"{df}: {len(hogs)}")
    print(f"{df} (filtered): {len(hogs_fltrd)}")

loss_fg: 215
loss_fg (filtered): 44
loss_bg: 406
loss_bg (filtered): 169
dup_fg: 464
dup_fg (filtered): 149
dup_bg: 514
dup_bg (filtered): 150


# PGLM

In [409]:
pglm_orb_hogs = list(pd.read_csv(f"{repo_root}/results/phyloglm/phyloglm_sig_hogs_positive.csv")["HOG"])
pglm_non_orb_hogs = list(pd.read_csv(f"{repo_root}/results/phyloglm/phyloglm_sig_hogs_negative.csv")["HOG"])

len(pglm_orb_hogs), len(pglm_non_orb_hogs)

(159, 178)

In [410]:
pglm_orb_udiv_locs = N5_udiv_blasted_df.loc[pglm_orb_hogs, "Gene"].tolist()
pglm_non_orb_ptep_locs = N5_ptep_blasted_df.loc[pglm_non_orb_hogs, "Gene"].tolist()

len(pglm_orb_udiv_locs), len(pglm_non_orb_ptep_locs)

(159, 178)

In [411]:
pglm_orb_universe_hogs = dup_universe_hogs
pglm_non_orb_universe_hogs = dup_universe_hogs


## Generate U.div LOC lists

In [412]:
for hog_list in (
    "relaxed_universe", 
    "relaxed",
    "busted_ph_orb_universe", 
    "busted_ph_orb",
    "loss_bg_universe",
    "loss_bg",
    "loss_bg_fltrd",
    "dup_fg_universe",
    "dup_fg",
    "dup_fg_fltrd",
    "pglm_orb_universe",
    "pglm_orb"
    ):
    globals()[f"{hog_list}_udiv_locs"] = N5_udiv_blasted_df.loc[globals()[f"{hog_list}_hogs"], "Gene"].tolist()

## Generate P.tep LOC lists

In [413]:
for hog_list in (
    "intensified_universe", 
    "intensified",
    "busted_ph_non_orb_universe",
    "busted_ph_non_orb",
    "loss_fg_universe",
    "loss_fg",
    "loss_fg_fltrd",
    "dup_bg_universe",
    "dup_bg",
    "dup_bg_fltrd",
    "pglm_non_orb_universe",
    "pglm_non_orb",
    ):
    globals()[f"{hog_list}_ptep_locs"] = N5_ptep_blasted_df.loc[globals()[f"{hog_list}_hogs"], "Gene"].tolist()

In [414]:
loc_lists = (
    "relaxed_universe_udiv_locs",
    "relaxed_udiv_locs",
    "intensified_universe_ptep_locs",
    "intensified_ptep_locs",
    "busted_ph_orb_universe_udiv_locs",
    "busted_ph_orb_udiv_locs",
    "loss_bg_universe_udiv_locs",
    "loss_bg_udiv_locs",
    "loss_bg_fltrd_udiv_locs",
    "dup_fg_universe_udiv_locs",
    "dup_fg_udiv_locs",
    "dup_fg_fltrd_udiv_locs",
    "pglm_orb_universe_udiv_locs",
    "pglm_orb_udiv_locs",
    "busted_ph_non_orb_universe_ptep_locs",
    "busted_ph_non_orb_ptep_locs",
    "loss_fg_universe_ptep_locs",
    "loss_fg_ptep_locs",
    "loss_fg_fltrd_ptep_locs",
    "dup_bg_universe_ptep_locs",
    "dup_bg_ptep_locs",
    "dup_bg_fltrd_ptep_locs",
    "pglm_non_orb_universe_ptep_locs",
    "pglm_non_orb_ptep_locs"
)

for loc_list_name in loc_lists:
    remove_dup_and_no_hits(loc_list_name)

relaxed_universe_udiv_locs: 4355
relaxed_udiv_locs: 392
intensified_universe_ptep_locs: 4482
intensified_ptep_locs: 723
busted_ph_orb_universe_udiv_locs: 4334
busted_ph_orb_udiv_locs: 91
loss_bg_universe_udiv_locs: 5904
loss_bg_udiv_locs: 380
loss_bg_fltrd_udiv_locs: 158
dup_fg_universe_udiv_locs: 9207
dup_fg_udiv_locs: 430
dup_fg_fltrd_udiv_locs: 135
pglm_orb_universe_udiv_locs: 9207
pglm_orb_udiv_locs: 150
busted_ph_non_orb_universe_ptep_locs: 4420
busted_ph_non_orb_ptep_locs: 234
loss_fg_universe_ptep_locs: 6016
loss_fg_ptep_locs: 209
loss_fg_fltrd_ptep_locs: 43
dup_bg_universe_ptep_locs: 9507
dup_bg_ptep_locs: 503
dup_bg_fltrd_ptep_locs: 146
pglm_non_orb_universe_ptep_locs: 9507
pglm_non_orb_ptep_locs: 173


In [415]:
#Write LOC lists to text files for GO enrichment
for name in loc_lists:
    dir_name = name.replace("_udiv_locs", "").replace("_ptep_locs", "").replace("_universe", "").replace("_fltrd", "")
    os.makedirs(f"{repo_root}/results/significant_gene_id_lists/{dir_name}", exist_ok=True)
    with open(f"{repo_root}/results/significant_gene_id_lists/{dir_name}/{name}.txt", "w") as f:
        for item in globals()[name]:
            f.write(f"{item}\n")